# Analisi degli indici azionari: S&P 500 ed EURO STOXX 50

Analisi esplorativa della performance storica di due indici azionari globali:
- **S&P 500** (`^GSPC`): indice delle 500 maggiori aziende quotate negli USA
- **EURO STOXX 50** (`^STOXX50E`): indice delle 50 maggiori aziende dell'Eurozona

L'analisi copre rendimenti annuali, mensili, giornalieri e una comparazione diretta tra i due indici.

In [ ]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
sp500 = yf.download('^GSPC', start='2000-01-01')
euro50 = yf.download('^STOXX50E', start='2000-01-01')

In [ ]:
print("=== S&P 500 ===")
print(sp500.info())
print(sp500.head())

In [ ]:
print("=== EURO STOXX 50 ===")
print(euro50.info())
print(euro50.head())

## Preprocessing: pulizia date e colonne ausiliarie

I DataFrame scaricati con yfinance vengono preprocessati applicando i seguenti passaggi:
1. Flatten delle colonne MultiIndex → colonne semplici (`Close`, `High`, ecc.)
2. Reset dell'indice → `Date` diventa una colonna esplicita
3. Conversione `Date` in tipo `datetime`
4. Ordinamento per data crescente
5. Aggiunta colonne ausiliarie: `Year`, `Month`, `Weekday`
6. Verifica dei valori mancanti

In [ ]:
def preprocess(df):
    """Flatten MultiIndex columns, reset index, convert Date, add Year/Month/Weekday."""
    df = df.droplevel(level=1, axis=1)       # ('Close', '^GSPC') → 'Close'
    df = df.reset_index()                     # Date come colonna esplicita
    df['Date'] = pd.to_datetime(df['Date'])   # assicura tipo datetime
    df = df.sort_values('Date').reset_index(drop=True)
    df['Year']    = df['Date'].dt.year
    df['Month']   = df['Date'].dt.month
    df['Weekday'] = df['Date'].dt.day_name()
    return df

sp500 = preprocess(sp500)
euro50 = preprocess(euro50)

In [ ]:
print("=== S&P 500 — dopo preprocessing ===")
print(sp500.dtypes)
print(sp500[['Date', 'Close', 'Year', 'Month', 'Weekday']].head())
print("\nValori mancanti:\n", sp500.isnull().sum())

print("\n=== EURO STOXX 50 — dopo preprocessing ===")
print(euro50.dtypes)
print(euro50[['Date', 'Close', 'Year', 'Month', 'Weekday']].head())
print("\nValori mancanti:\n", euro50.isnull().sum())

## Rendimenti annuali e mensili

Per ogni indice vengono calcolati:
- **Rendimento annuale**: prezzo di chiusura dell'ultimo giorno di ogni anno → `pct_change()` tra anni consecutivi
- **Rendimento mensile**: stesso approccio, raggruppando per anno e mese

In [ ]:
def calc_annual_returns(df):
    """Rendimento annuale: ultimo Close di ogni anno → pct_change tra anni."""
    annual = df.groupby('Year')['Close'].last()
    returns = annual.pct_change().dropna() * 100
    returns.name = 'Annual_Return_%'
    return returns

def calc_monthly_returns(df):
    """Rendimento mensile: ultimo Close di ogni mese → pct_change tra mesi consecutivi."""
    monthly = df.groupby(['Year', 'Month'])['Close'].last()
    returns = monthly.pct_change().dropna() * 100
    returns.name = 'Monthly_Return_%'
    return returns

In [ ]:
sp500_annual  = calc_annual_returns(sp500)
euro50_annual = calc_annual_returns(euro50)

print("=== Rendimenti annuali S&P 500 ===")
print(sp500_annual.round(2).to_string())
print(f"\nMedia: {sp500_annual.mean():.2f}%  |  Migliore: {sp500_annual.idxmax()} ({sp500_annual.max():.2f}%)  |  Peggiore: {sp500_annual.idxmin()} ({sp500_annual.min():.2f}%)")

print("\n=== Rendimenti annuali EURO STOXX 50 ===")
print(euro50_annual.round(2).to_string())
print(f"\nMedia: {euro50_annual.mean():.2f}%  |  Migliore: {euro50_annual.idxmax()} ({euro50_annual.max():.2f}%)  |  Peggiore: {euro50_annual.idxmin()} ({euro50_annual.min():.2f}%)")

In [ ]:
sp500_monthly  = calc_monthly_returns(sp500)
euro50_monthly = calc_monthly_returns(euro50)

print("=== Rendimenti mensili S&P 500 (ultimi 12 mesi) ===")
print(sp500_monthly.tail(12).round(2).to_string())

print("\n=== Rendimenti mensili EURO STOXX 50 (ultimi 12 mesi) ===")
print(euro50_monthly.tail(12).round(2).to_string())

## Rendimenti giornalieri e stagionalità settimanale

Per ogni indice vengono calcolati:
- **Rendimento giornaliero**: variazione percentuale del Close giorno per giorno (`pct_change()`)
- **Stagionalità settimanale**: rendimento medio per giorno della settimana, per identificare quale giorno rende di più storicamente
- **Volume medio** per giorno della settimana

In [ ]:
def calc_weekday_returns(df):
    """Rendimento medio e volume medio per giorno della settimana."""
    df = df.copy()
    df['Daily_Return'] = df['Close'].pct_change() * 100

    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']

    avg_return = (
        df.groupby('Weekday')['Daily_Return']
        .mean()
        .reindex(weekday_order)
    )
    avg_volume = (
        df.groupby('Weekday')['Volume']
        .mean()
        .reindex(weekday_order)
    )
    return df, avg_return, avg_volume

In [ ]:
sp500,  sp500_weekday,  sp500_vol  = calc_weekday_returns(sp500)
euro50, euro50_weekday, euro50_vol = calc_weekday_returns(euro50)

print("=== S&P 500 — rendimento medio % per giorno ===")
print(sp500_weekday.round(4).to_string())
print(f"Giorno migliore: {sp500_weekday.idxmax()} ({sp500_weekday.max():.4f}%)")
print(f"Giorno peggiore: {sp500_weekday.idxmin()} ({sp500_weekday.min():.4f}%)")

print("\n=== S&P 500 — volume medio per giorno ===")
print(sp500_vol.apply(lambda x: f"{x:,.0f}").to_string())

print("\n=== EURO STOXX 50 — rendimento medio % per giorno ===")
print(euro50_weekday.round(4).to_string())
print(f"Giorno migliore: {euro50_weekday.idxmax()} ({euro50_weekday.max():.4f}%)")
print(f"Giorno peggiore: {euro50_weekday.idxmin()} ({euro50_weekday.min():.4f}%)")

print("\n=== EURO STOXX 50 — volume medio per giorno ===")
print(euro50_vol.apply(lambda x: f"{x:,.0f}").to_string())

## Refactoring: funzioni riutilizzabili

Le funzioni scritte nelle sezioni precedenti vengono qui consolidate in un'API pulita, con l'aggiunta di due nuove funzioni:
- `download_index(ticker, name)` — scarica + preprocessa in un solo passo
- `summary_stats(df, name)` — statistiche riassuntive (best/worst day, volume medio)

L'intera analisi è quindi rieseguibile con poche righe, applicando le stesse funzioni a entrambi gli indici.

In [ ]:
def download_index(ticker, name):
    """Scarica i dati storici e applica il preprocessing completo."""
    df = yf.download(ticker, start='2000-01-01', progress=False)
    df = df.droplevel(level=1, axis=1)
    df = df.reset_index()
    df['Date']    = pd.to_datetime(df['Date'])
    df            = df.sort_values('Date').reset_index(drop=True)
    df['Year']    = df['Date'].dt.year
    df['Month']   = df['Date'].dt.month
    df['Weekday'] = df['Date'].dt.day_name()
    df['Daily_Return'] = df['Close'].pct_change() * 100
    df.name = name
    return df

def summary_stats(df, name):
    """Stampa best/worst day e volume medio per giorno della settimana."""
    weekday_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday']
    avg_return = df.groupby('Weekday')['Daily_Return'].mean().reindex(weekday_order)
    avg_volume = df.groupby('Weekday')['Volume'].mean().reindex(weekday_order)

    print(f"=== {name} ===")
    print(f"  Giorno migliore : {avg_return.idxmax()} ({avg_return.max():.4f}%)")
    print(f"  Giorno peggiore : {avg_return.idxmin()} ({avg_return.min():.4f}%)")
    print(f"  Volume medio    : {avg_volume.mean():,.0f}")
    return avg_return, avg_volume

In [ ]:
# Con le funzioni definite sopra, l'intera analisi si riduce a poche righe
sp500  = download_index('^GSPC',     'S&P 500')
euro50 = download_index('^STOXX50E', 'EURO STOXX 50')

sp500_annual   = calc_annual_returns(sp500)
euro50_annual  = calc_annual_returns(euro50)

sp500_monthly  = calc_monthly_returns(sp500)
euro50_monthly = calc_monthly_returns(euro50)

sp500_weekday,  sp500_vol  = summary_stats(sp500,  'S&P 500')
euro50_weekday, euro50_vol = summary_stats(euro50, 'EURO STOXX 50')

## Visualizzazioni

I risultati dell'analisi vengono rappresentati graficamente attraverso quattro tipi di visualizzazione:
1. **Bar chart** rendimenti annuali per ciascun indice
2. **Heatmap** rendimenti mensili (anno × mese)
3. **Istogramma** distribuzione dei rendimenti giornalieri
4. **Bar chart** rendimento medio per giorno della settimana

In [ ]:
# Bar chart — rendimenti annuali
fig, axes = plt.subplots(2, 1, figsize=(14, 10))

for ax, returns, title in zip(
    axes,
    [sp500_annual, euro50_annual],
    ['S&P 500 — Rendimenti annuali (%)', 'EURO STOXX 50 — Rendimenti annuali (%)']
):
    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in returns]
    ax.bar(returns.index, returns.values, color=colors, edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Anno')
    ax.set_ylabel('Rendimento (%)')
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Heatmap — rendimenti mensili (anno × mese)
fig, axes = plt.subplots(2, 1, figsize=(14, 12))

month_labels = ['Gen', 'Feb', 'Mar', 'Apr', 'Mag', 'Giu',
                'Lug', 'Ago', 'Set', 'Ott', 'Nov', 'Dic']

for ax, monthly, title in zip(
    axes,
    [sp500_monthly, euro50_monthly],
    ['S&P 500 — Rendimenti mensili (%)', 'EURO STOXX 50 — Rendimenti mensili (%)']
):
    pivot = monthly.unstack(level='Month')
    pivot.columns = month_labels[:len(pivot.columns)]
    sns.heatmap(
        pivot, ax=ax, cmap='RdYlGn', center=0,
        annot=True, fmt='.1f', annot_kws={'size': 7},
        linewidths=0.3, cbar_kws={'label': 'Rendimento (%)'}
    )
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Mese')
    ax.set_ylabel('Anno')

plt.tight_layout()
plt.show()

In [ ]:
# Istogramma — distribuzione rendimenti giornalieri
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, df, title, color in zip(
    axes,
    [sp500, euro50],
    ['S&P 500 — Distribuzione rendimenti giornalieri',
     'EURO STOXX 50 — Distribuzione rendimenti giornalieri'],
    ['#3498db', '#9b59b6']
):
    daily = df['Daily_Return'].dropna()
    ax.hist(daily, bins=100, color=color, alpha=0.75, edgecolor='white', linewidth=0.3)
    ax.axvline(0, color='black', linewidth=1)
    ax.axvline(daily.mean(), color='red', linewidth=1.2, linestyle='--',
               label=f'Media: {daily.mean():.3f}%')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Rendimento giornaliero (%)')
    ax.set_ylabel('Frequenza')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Bar chart — rendimento medio per giorno della settimana
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, avg_return, title in zip(
    axes,
    [sp500_weekday, euro50_weekday],
    ['S&P 500 — Rendimento medio per giorno della settimana',
     'EURO STOXX 50 — Rendimento medio per giorno della settimana']
):
    colors = ['#2ecc71' if v >= 0 else '#e74c3c' for v in avg_return]
    ax.bar(avg_return.index, avg_return.values, color=colors, edgecolor='white')
    ax.axhline(0, color='black', linewidth=0.8)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('Giorno della settimana')
    ax.set_ylabel('Rendimento medio (%)')
    ax.tick_params(axis='x', rotation=20)

plt.tight_layout()
plt.show()

## Analisi comparativa: S&P 500 vs EURO STOXX 50

I due indici vengono confrontati direttamente attraverso tre visualizzazioni:
1. **Line chart** performance normalizzata a 100 dalla data comune di inizio
2. **Bar chart** rendimenti annuali affiancati per anno
3. **Scatter plot** correlazione tra i rendimenti giornalieri, con calcolo di `corr()` e `std()`

In [ ]:
# Line chart — performance normalizzata a 100 dalla data comune di inizio
common_start = max(sp500['Date'].min(), euro50['Date'].min())

sp500_common  = sp500[sp500['Date'] >= common_start].set_index('Date')['Close']
euro50_common = euro50[euro50['Date'] >= common_start].set_index('Date')['Close']

sp500_norm  = sp500_common  / sp500_common.iloc[0]  * 100
euro50_norm = euro50_common / euro50_common.iloc[0] * 100

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(sp500_norm.index,  sp500_norm.values,  label='S&P 500',       color='#3498db', linewidth=1.2)
ax.plot(euro50_norm.index, euro50_norm.values, label='EURO STOXX 50', color='#e67e22', linewidth=1.2)
ax.axhline(100, color='gray', linewidth=0.7, linestyle='--')
ax.set_title('Performance relativa normalizzata a 100', fontsize=13, fontweight='bold')
ax.set_xlabel('Data')
ax.set_ylabel('Valore (base 100)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bar chart — rendimenti annuali affiancati
common_years = sorted(set(sp500_annual.index) & set(euro50_annual.index))
sp500_common_annual  = sp500_annual.loc[common_years]
euro50_common_annual = euro50_annual.loc[common_years]

x = range(len(common_years))
width = 0.4

fig, ax = plt.subplots(figsize=(14, 5))
ax.bar([i - width/2 for i in x], sp500_common_annual.values,  width=width, label='S&P 500',       color='#3498db', alpha=0.85)
ax.bar([i + width/2 for i in x], euro50_common_annual.values, width=width, label='EURO STOXX 50', color='#e67e22', alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(list(x))
ax.set_xticklabels(common_years, rotation=45)
ax.set_title('Rendimenti annuali affiancati: S&P 500 vs EURO STOXX 50', fontsize=13, fontweight='bold')
ax.set_xlabel('Anno')
ax.set_ylabel('Rendimento (%)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plot correlazione + statistiche
merged = sp500.set_index('Date')[['Daily_Return']].join(
    euro50.set_index('Date')[['Daily_Return']],
    how='inner', lsuffix='_sp500', rsuffix='_euro50'
).dropna()

corr = merged['Daily_Return_sp500'].corr(merged['Daily_Return_euro50'])
std_sp500  = merged['Daily_Return_sp500'].std()
std_euro50 = merged['Daily_Return_euro50'].std()

fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(merged['Daily_Return_sp500'], merged['Daily_Return_euro50'],
           alpha=0.15, s=5, color='#2c3e50')
ax.axhline(0, color='gray', linewidth=0.6)
ax.axvline(0, color='gray', linewidth=0.6)
ax.set_title(f'Correlazione rendimenti giornalieri\nr = {corr:.3f}', fontsize=13, fontweight='bold')
ax.set_xlabel('S&P 500 — Rendimento giornaliero (%)')
ax.set_ylabel('EURO STOXX 50 — Rendimento giornaliero (%)')
plt.tight_layout()
plt.show()

print(f"Correlazione (r):        {corr:.3f}")
print(f"Volatilità S&P 500:      {std_sp500:.3f}%  (std rendimenti giornalieri)")
print(f"Volatilità EURO STOXX:   {std_euro50:.3f}%  (std rendimenti giornalieri)")

## Conclusioni

L'analisi esplorativa dei dati storici di S&P 500 ed EURO STOXX 50 ha prodotto i seguenti risultati principali:

**Performance storica**
- L'S&P 500 ha registrato una crescita superiore rispetto all'EURO STOXX 50 nel periodo comune (dal 2007), grazie in particolare alla decade 2010–2020 dominata dalle grandi tech americane.
- Entrambi gli indici hanno subito crolli significativi negli stessi anni (2008, 2020), confermando la loro esposizione ai cicli economici globali.

**Rendimenti annuali e mensili**
- I rendimenti annuali mostrano un'elevata variabilità: anni con +25%/+30% si alternano ad anni con -30%/−40%.
- La heatmap mensile non evidenzia pattern stagionali forti e consistenti nel tempo.

**Stagionalità settimanale**
- Le differenze di rendimento medio tra i giorni della settimana sono nell'ordine di pochi centesimi di punto percentuale — statisticamente rilevabili sui 25 anni di dati, ma troppo piccole per avere significato pratico.

**Correlazione e volatilità**
- I due indici mostrano una correlazione significativa nei rendimenti giornalieri, segno che i mercati USA ed europei reagiscono agli stessi shock globali.
- L'EURO STOXX 50 presenta una volatilità giornaliera leggermente superiore all'S&P 500.